In [0]:
from __future__ import annotations
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.types import StructType


class CSVReader:


    def __init__(self, catalog: str, schema: str,volume: str, header: bool = True, infer_schema: bool = True, sep: str = ",", encoding: str = "UTF-8", **default_options) -> None:
        self.catalog = catalog
        self.schema = schema
        self.volume = volume
        self.spark = (
            spark
            or SparkSession.getActiveSession()
            or SparkSession.builder.getOrCreate()
        )
        self.default_options: dict = {
            "header": header,
            "inferSchema": infer_schema,
            "sep": sep,
            "encoding": encoding,
            **default_options,
        }

    @property
    def base_path(self) -> str:
        return f"/Volumes/{self.catalog}/{self.schema}/{self.volume}"

    def _resolve_path(self, filename: str) -> str:
        """Monta o caminho absoluto do arquivo dentro do volume."""
        return f"{self.base_path}/{filename.lstrip('/')}"

    @staticmethod
    def _normalize(options: dict) -> dict:
        return {
            key: str(value).lower() if isinstance(value, bool) else value
            for key, value in options.items()
        }

    def read(self,filename: str,schema: StructType | None = None,**options) -> DataFrame:

        path = self._resolve_path(filename)
        merged = self._normalize({**self.default_options, **options})

        reader = self.spark.read.format("csv").options(**merged)

        if schema is not None:
            merged.pop("inferSchema", None)
            reader = self.spark.read.format("csv").options(**merged).schema(schema)

        return reader.load(path)

    def __repr__(self) -> str:
        return (
            f"VolumeCSVReader(base_path='{self.base_path}', "
            f"options={self.default_options})"
        )

In [0]:
reader = CSVReader(
            catalog="unifor_ed_t2",
            schema="landing",
            volume="olist",
            sep=",",
        )
df = reader.read("olist_customers_dataset.csv")
df.display()